<a href="https://colab.research.google.com/github/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/blob/main/code/NRE5615_Wk9_Demo_CameraTrap_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 Demo: Camera Trap Image Classification

In this demo, we train a simple image classification model using a small camera trap dataset. The images are grouped into three classes:

- `animal`
- `empty`
- `human_vehicle`

The goal is to practice a basic AI workflow: upload data, inspect examples, train a model, test predictions, and interpret results. This is a teaching example, not a production wildlife monitoring model.

## Dataset format

Upload one ZIP file with this folder structure:

```text
Week9_CameraTrap_TeachableMachine/
├── train/
│   ├── animal/
│   ├── empty/
│   └── human_vehicle/
└── test/
    ├── animal/
    ├── empty/
    └── human_vehicle/
```

The ZIP file should be small enough to upload directly into Colab.

## Step 1: Upload and unzip the dataset

Run this cell and upload the camera trap ZIP file when prompted.

In [ ]:
#from google.colab import files
import requests
import zipfile
from pathlib import Path

url = "https://raw.githubusercontent.com/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/main/data/Wk9_Camera_Trap_Data_Sample.zip"

zip_name = "Wk9_Camera_Trap_Data_Sample.zip"

# Download the zip file
response = requests.get(url)
response.raise_for_status()

with open(zip_name, "wb") as f:
    f.write(response.content)

print("Downloaded:", zip_name)


DATA_DIR = Path("/content/camera_trap_data")

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Dataset extracted to:", DATA_DIR)

!find /content/camera_trap_data -maxdepth 3 -type d

## Step 2: Locate the train and test folders

This cell finds the folder that contains the `train` and `test` image folders.

In [ ]:
possible_train_folders = list(DATA_DIR.rglob("train"))

if len(possible_train_folders) == 0:
    raise FileNotFoundError("Could not find a folder named 'train'. Check the ZIP file structure.")

train_dir = possible_train_folders[0]
root_dir = train_dir.parent
test_dir = root_dir / "test"

if not test_dir.exists():
    raise FileNotFoundError("Could not find a folder named 'test' next to the train folder.")

print("Root folder:", root_dir)
print("Train folder:", train_dir)
print("Test folder:", test_dir)

## Step 3: Count images in each class

A balanced teaching dataset should have a similar number of images in each class.

In [ ]:
for split_dir in [train_dir, test_dir]:
    print(split_dir.name.upper())
    for class_folder in sorted(split_dir.iterdir()):
        if class_folder.is_dir():
            image_files = [p for p in class_folder.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            print(f"  {class_folder.name}: {len(image_files)}")

## Step 4: Preview example images

Look at a few images from each class before training the model.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
print("Classes:", class_names)

plt.figure(figsize=(9, 6))

plot_num = 1
for class_name in class_names:
    image_files = [p for p in (train_dir / class_name).glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    sample_files = random.sample(image_files, min(3, len(image_files)))

    for img_path in sample_files:
        img = Image.open(img_path)
        plt.subplot(len(class_names), 3, plot_num)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis("off")
        plot_num += 1

plt.tight_layout()
plt.show()

## Step 5: Load images for model training

The images are resized to 160 × 160 pixels for the model. This keeps the demo fast and simple.

In [ ]:
import tensorflow as tf

IMG_SIZE = (160, 160)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## Step 6: Build and train a simple image classifier

This demo uses **transfer learning** with MobileNetV2. The model has already learned useful image patterns from many general images. We freeze that part of the model and train a small classification layer for our camera trap classes.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

## Step 7: Visualize training and test accuracy

This plot shows whether the model improved during training and how it performed on the test set.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Test accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Camera Trap Image Classification")
plt.legend()
plt.show()

## Step 8: Evaluate the model

The test accuracy is calculated using images that were not used for model training.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test accuracy:", round(test_accuracy, 3))

## Step 9: Show example predictions

Review examples where the model was correct and where it was wrong or uncertain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

for images, labels in test_ds.take(1):
    predictions = model.predict(images)

    for i in range(min(9, len(images))):
        true_label = class_names[np.argmax(labels[i])]
        predicted_label = class_names[np.argmax(predictions[i])]
        confidence = np.max(predictions[i])

        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"True: {true_label}\nPred: {predicted_label} ({confidence:.2f})")
        plt.axis("off")

plt.tight_layout()
plt.show()


## Step 10: Show confusion matrix for test set

Review the overal model perfomance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Get true labels and predicted labels
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

# Create and show confusion matrix
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot()
plt.title("Confusion Matrix: Camera Trap Test Set")
plt.show()

## Reflection questions

1. Which class seemed easiest for the model to classify?
2. Which class seemed most difficult?
3. Did the model make any mistakes? What might explain those mistakes?
4. How could this workflow help with wildlife or biodiversity monitoring?
5.	What are two limitations or risks of using an AI model for camera trap image classification?
6. Why should humans still review AI results before using them for conservation decisions?

## Optional Guide: Using Google Teachable Machine with the Camera Trap Dataset

Google Teachable Machine is a beginner-friendly tool for training a simple AI image classifier without writing code. In this activity, you will use the same camera trap dataset from this notebook.

The dataset has three classes:

- `animal`
- `empty`
- `human_vehicle`

The goal is to train a model that can sort camera trap images into these three categories.

---

## Step 1: Download the Dataset

Use the course-provided camera trap dataset zip file.

The folder should contain:

```text
train/
  animal/
  empty/
  human_vehicle/

test/
  animal/
  empty/
  human_vehicle/

### Step 2: Open Google Teachable Machine

Go to Google Teachable Machine:

https://teachablemachine.withgoogle.com/

Click:

Get Started

Then choose:

Image Project

Then choose:

Standard image model

### Step 3: Create the Three Classes

You will see a page with image classes.

Rename the first class:

```animal```

Rename the second class:

```empty```

Add a third class and name it:

```human_vehicle```

### Step 4: Upload Training Images

For each class, upload the matching images from the train folder.

Use:
[Linkt to zip file with data](https://raw.githubusercontent.com/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/main/data/Wk9_Camera_Trap_Data_Sample.zip)


Upload → Choose images from your files


Upload images as follows:
```
train/animal          → animal class
train/empty           → empty class
train/human_vehicle   → human_vehicle class
```

### Step 5: Train the Model

After all training images are uploaded, click:

```Train Model```

Wait until training finishes.

Do not close the browser tab while the model is training.

### Step 6: Test the Model

After training, use the preview panel to test the model.

Upload images from the test folder:
```
test/animal
test/empty
test/human_vehicle
```
Try several images from each class.

For each test image, observe:

- the predicted class,
- the confidence score,
- whether the prediction is correct or incorrect.